In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import transformers
import os
import sys
import json
import matplotlib.pyplot as plt
from transformers import pipeline, AutoTokenizer

The aim is to fine-tune the TinyLlama model to be able to answer questions relating to medical problems more accurately.

In [ ]:
df = pd.read_parquet("hf://datasets/lavita/ChatDoctor-HealthCareMagic-100k/data/train-00000-of-00001-5e7cb295b9cff0bf.parquet")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [ ]:
def format_prompt(example):
    chat = example
    prompt = tokenizer.apply_chat_template(chat, tokenize=False)
    return prompt

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

# 1. Load Model and Tokenizer
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # or your saved model path
model = AutoModelForCausalLM.from_pretrained(model_name)

# 2. Define Prompt Formatting Function (from your code)
def format_prompt(example):
    chat = example
    prompt = tokenizer.apply_chat_template(chat, tokenize=False)
    return prompt

# 3. Function to Ask the Model a Question
def ask_model(question, chat_history=[]):
    """Asks the loaded model a question.

    Args:
        question: The question to ask the model.
        chat_history: (Optional) A list of previous messages for context.

    Returns:
        The model's response as a string.
    """

    new_chat_history = chat_history + [{"role": "user", "content": question}]
    input_ids = tokenizer(format_prompt(new_chat_history), return_tensors="pt").input_ids

    # Generate Response
    with torch.no_grad():  # Disable gradient calculation during inference
        generated_ids = model.generate(input_ids)

    response = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    return response

# 4. Example Usage
question = df["instruction"][0] + " " + df["input"][0]
chat_history = [
    {"role": "user", "content": "You are a doctor, answer the medical questions based on the patient's description:"},
    {"role": "assistant", "content": "Okay, I'm ready to help."}
]

response = ask_model(question, chat_history)
print(response)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 15.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [ ]:
from datasets import load_dataset

In [ ]:
dataset = (
    load_dataset("HuggingFaceH4/ultrachat_200k", split="test_sft").shuffle(seed=42).select(range(3_000))
    )

README.md:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

(…)-00000-of-00003-a3ecf92756993583.parquet:   0%|          | 0.00/244M [00:00<?, ?B/s]

(…)-00001-of-00003-0a1804bcb6ae68c6.parquet:   0%|          | 0.00/244M [00:00<?, ?B/s]

(…)-00002-of-00003-ee46ed25cfae92c6.parquet:   0%|          | 0.00/244M [00:00<?, ?B/s]

(…)-00000-of-00001-f7dfac4afe5b93f4.parquet:   0%|          | 0.00/81.2M [00:00<?, ?B/s]

...

(…)-00000-of-00001-3d4cd8309148a71f.parquet:   0%|          | 0.00/80.4M [00:00<?, ?B/s]

Generating train_sft split:   0%|          | 0/207865 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/23110 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/256032 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/28304 [00:00<?, ? examples/s]

In [ ]:
print(type(dataset))

<class 'datasets.arrow_dataset.Dataset'>


In [ ]:
print(format_prompt(dataset["messages"][2000]))

<|user|>
Provide a vivid description of a bustling marketplace filled with countless vendors selling a colorful array of freshly prepared foods and artisanal crafts that are handmade and unique to the region. Use sensory language to convey the sights, sounds, smells, and overall atmosphere of the market.</s>
<|assistant|>
As soon as you step foot into the bustling marketplace, your senses are overwhelmed with an incredible array of colors, aromas, and sounds that are impossible to ignore. The vibrant mixture of smells and sounds flood your senses giving you an instant feeling of excitement and anticipation.

The vibrant colors of the artisanal crafts and freshly prepared foods catch your eye as you wander through the maze of vendors selling everything from handmade jewelry to intricately woven textiles. The bright reds, blues, and yellows of the hand-painted pottery seem to shimmer in the sunlight, encouraging you to pick one up and feel its smooth, cool curves in your hands.

As you n

In [ ]:
from pprint import pprint
pprint(dataset["messages"][2000])

[{'content': 'Provide a vivid description of a bustling marketplace filled '
             'with countless vendors selling a colorful array of freshly '
             'prepared foods and artisanal crafts that are handmade and unique '
             'to the region. Use sensory language to convey the sights, '
             'sounds, smells, and overall atmosphere of the market.',
  'role': 'user'},
 {'content': 'As soon as you step foot into the bustling marketplace, your '
             'senses are overwhelmed with an incredible array of colors, '
             'aromas, and sounds that are impossible to ignore. The vibrant '
             'mixture of smells and sounds flood your senses giving you an '
             'instant feeling of excitement and anticipation.\n'
             '\n'
             'The vibrant colors of the artisanal crafts and freshly prepared '
             'foods catch your eye as you wander through the maze of vendors '
             'selling everything from handmade jewelry 

In [ ]:
df.head()

,instruction,input,output
0,"If you are a doctor, please answer the medical...",I woke up this morning feeling the whole room ...,"Hi, Thank you for posting your query. The most..."
1,"If you are a doctor, please answer the medical...",My baby has been pooing 5-6 times a day for a ...,Hi... Thank you for consulting in Chat Doctor....
2,"If you are a doctor, please answer the medical...","Hello, My husband is taking Oxycodone due to a...","Hello, and I hope I can help you today.First, ..."
3,"If you are a doctor, please answer the medical...",lump under left nipple and stomach pain (male)...,HI. You have two different problems. The lump ...
4,"If you are a doctor, please answer the medical...",I have a 5 month old baby who is very congeste...,Thank you for using Chat Doctor. I would sugge...


In [ ]:
instruction = "You are a doctor, answer the medical questions based on the patient's description: "

In [ ]:
def parse_input(text):
  return {"content":instruction + text, "role":"user"}

def parse_output(text):
  return {"content":text, "role":"assistant"}

In [ ]:
df['input'][2000]

'my mothers age 58, jaundice problemcreatinine 4.88, sodium-125, potasium-4.6; chloride-85;bilirubin total-17.16;bilirubin direct-2.38;sgot-155;sgpt-160;alkaline phos-260;protein-7.54;albumin-3.4;urea-138 and BP 30nowplease help. what is status and in which stage she is?'

In [ ]:
count = 0
for i, j in zip(df["input"], df["output"]):
  df.loc[count, "input"] = str(parse_input(i))
  df.loc[count, "output"] = str(parse_output(j))
  count += 1

In [ ]:
ex = df["input"][2000]

In [ ]:
import ast

In [ ]:
lit_ = lambda x: ast.literal_eval(x)

In [ ]:
new_df = []
for i, j  in zip(df["input"], df["output"]):
  new_df.append([lit_(i), lit_(j)])

In [ ]:
train_df = new_df[:5000]

In [ ]:
def format_prompt(example):
  chat = example
  prompt = tokenizer.apply_chat_template(chat, tokenize=False)
  return prompt

In [ ]:
train_df = [format_prompt(prompt) for prompt in train_df]
train_col = list(range(len(train_df)))
data = {"train":train_col, "text":train_df}
train_df = pd.DataFrame(data)

train_df.to_csv("train.csv", index=False)

In [ ]:
train_df = load_dataset('csv', data_files='train.csv')

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
train_df

DatasetDict({
    train: Dataset({
        features: ['train', 'text'],
        num_rows: 5000
    })
})

In [ ]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 MB 8.3 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments

model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True
)

model.config.use_cache = False
model.config.pretraining_tp = 1

t_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = "<PAD>"
tokenizer.padding_side = "left"


config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
peft_config = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.1,
    r=16,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ]
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

In [ ]:
output_dir = "./results"

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate = 2e-4,
    lr_scheduler_type = "cosine",
    num_train_epochs=1,
    logging_steps=10,
    fp16=True,
    gradient_checkpointing=True
)

In [ ]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.9/310.9 kB 15.7 MB/s eta 0:00:00


In [ ]:
from trl import SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_df,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
)

trainer.train()
trainer.model.save_pretrained("TinyLlama-1.1B-medqa-qlora")

/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:300: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:328: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:403: UserWarning: You passed a processing_class with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `processing_class.padding_side = 'right'` to your code.
  warnings.warn(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

 ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


KeyError: "Invalid key: 0. Please first select a split. For example: `my_dataset_dictionary['train'][0]`. Available splits: ['train']"